In [4]:
!pip install -q scikit-learn joblib streamlit colabcode

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from joblib import dump
import numpy as np
import os

DATASET_NAME = "realty_data"
MODEL_FILENAME = "realty_data_model.pkl"
FEATURE_NAMES_FILENAME = "realty_data_feature_names.txt"

# Загрузка данных
data = fetch_california_housing()
X = data.data
y = data.target
feature_names = data.feature_names

# Разделение
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Обучение
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

preds = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, preds))  # <- здесь совместимо со всеми версиями
print(f"Validation RMSE ({DATASET_NAME}) (в единицах 100k USD): {rmse:.4f}")

# Обучение на всей выборке и сохранение
model.fit(X, y)
dump(model, MODEL_FILENAME)

with open(FEATURE_NAMES_FILENAME, "w") as f:
    f.write(",".join(feature_names))

print(f"Model saved: {MODEL_FILENAME}")
print(f"Feature names saved: {FEATURE_NAMES_FILENAME}")

import streamlit as st
import numpy as np
from joblib import load
import os

MODEL_PATH = "realty_data_model.pkl"

@st.cache(allow_output_mutation=True)
def load_model():
    if not os.path.exists(MODEL_PATH):
        st.error("Модель не найдена. Выполните обучение (train) перед запуском приложения.")
        return None
    return load(MODEL_PATH)

def main():
    st.set_page_config(page_title="Прогноз цены недвижимости", layout="centered")
    st.title("Прогноз медианной стоимости домов (RealtyData)")
    st.write("Вводите значения признаков и нажмите Predict. Результат — в долларах США.")

    st.markdown("### Введите признаки (в той же последовательности, что и в датасете):")
    col1, col2 = st.columns(2)
    with col1:
        med_inc = st.number_input("MedInc (Median Income)", value=6.0, min_value=0.0, step=0.1)
        house_age = st.number_input("HouseAge (Возраст домов)", value=28.0, min_value=0.0, step=1.0)
        ave_rooms = st.number_input("AveRooms (Среднее число комнат)", value=6.0, min_value=0.0, step=0.1)
        ave_bedrooms = st.number_input("AveBedrooms (Среднее число спален)", value=1.0, min_value=0.0, step=0.1)
    with col2:
        population = st.number_input("Population (Население округа)", value=1500.0, min_value=0.0, step=1.0)
        ave_occup = st.number_input("AveOccup (Средняя занятость)", value=2.5, min_value=0.0, step=0.1)
        latitude = st.number_input("Latitude", value=34.05, min_value=-90.0, max_value=90.0, step=0.01)
        longitude = st.number_input("Longitude", value=-118.25, min_value=-180.0, max_value=180.0, step=0.01)

    if st.button("Predict"):
        model = load_model()
        if model is None:
            return
        features = np.array([[med_inc, house_age, ave_rooms, ave_bedrooms, population, ave_occup, latitude, longitude]])
        pred = model.predict(features)[0]
        pred_dollars = pred * 100000
        st.success(f"Predicted median house value: ${pred_dollars:,.0f}")
        st.write(f"(Модель прогнозирует {pred:.3f} в единицах 100k USD.)")

if __name__ == "__main__":
    main()

app_code = r"""
import streamlit as st
import numpy as np
from joblib import load
import os

MODEL_PATH = "realty_data_model.pkl"

@st.cache_resource
def load_model():
    if not os.path.exists(MODEL_PATH):
        st.error("Модель не найдена. Выполните обучение (train) перед запуском приложения.")
        return None
    return load(MODEL_PATH)

def main():
    st.set_page_config(page_title="Прогноз цены недвижимости", layout="centered")
    st.title("Прогноз медианной стоимости домов (RealtyData)")
    st.write("Вводите значения признаков и нажмите Predict. Результат — в долларах США.")

    st.markdown("### Введите признаки (в той же последовательности, что и в датасете):")
    col1, col2 = st.columns(2)
    with col1:
        med_inc = st.number_input("MedInc (Median Income)", value=6.0, min_value=0.0, step=0.1)
        house_age = st.number_input("HouseAge (Возраст домов)", value=28.0, min_value=0.0, step=1.0)
        ave_rooms = st.number_input("AveRooms (Среднее число комнат)", value=6.0, min_value=0.0, step=0.1)
        ave_bedrooms = st.number_input("AveBedrooms (Среднее число спален)", value=1.0, min_value=0.0, step=0.1)
    with col2:
        population = st.number_input("Population (Население округа)", value=1500.0, min_value=0.0, step=1.0)
        ave_occup = st.number_input("AveOccup (Средняя занятость)", value=2.5, min_value=0.0, step=0.1)
        latitude = st.number_input("Latitude", value=34.05, min_value=-90.0, max_value=90.0, step=0.01)
        longitude = st.number_input("Longitude", value=-118.25, min_value=-180.0, max_value=180.0, step=0.01)

    if st.button("Predict"):
        model = load_model()
        if model is None:
            return
        features = np.array([[med_inc, house_age, ave_rooms, ave_bedrooms, population, ave_occup, latitude, longitude]])
        pred = model.predict(features)[0]  # в единицах 100k USD
        pred_dollars = pred * 100000
        st.success(f"Predicted median house value: ${pred_dollars:,.0f}")
        st.write(f"(Модель прогнозирует {pred:.3f} в единицах 100k USD.)")

if __name__ == "__main__":
    main()
"""
open("app.py", "w", encoding="utf-8").write(app_code)
print("app.py создан.")

Validation RMSE (realty_data) (в единицах 100k USD): 0.5053
Model saved: realty_data_model.pkl
Feature names saved: realty_data_feature_names.txt


2026-02-25 17:48:00.098 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-25 17:48:00.295 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-02-25 17:48:00.297 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-25 17:48:00.299 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-25 17:48:00.300 
`st.cache` is deprecated and will be removed soon. Please use one of Streamlit's new
caching commands, `st.cache_data` or `st.cache_resource`. More information
[in our docs](https://docs.streamlit.io/develop/concepts/architecture/caching).

**Note**: The behavior of `st.cache` was updated in Streamlit 1.36 to the new caching
logic used by `st.cache_data` and `st.cache_resource`. This might lead to some problems
or unexpected behavior in certain edge cases.

202

app.py создан.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')